In [5]:
from typing import Annotated
from langchain_ollama import ChatOllama
from langchain_core.messages import AnyMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage

In [2]:
load_dotenv()

True

In [3]:
llm = ChatOllama(
    model="gemma4:e2b",
    temperature=0.7
)

In [6]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):
    decision = interrupt({
        "type": "approval",
        "reason": "Model is about to answer a user question.",
        "question": state["messages"][-1].content,
        "instruction": "Approve this question? yes/no"
    })
    
    if decision["approved"] == 'no':
        return {"messages": [AIMessage(content="Not approved.")]}

    else:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

In [11]:
builder = StateGraph(ChatState)

builder.add_node("chat", chat_node)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)
checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

In [ ]:
config = {"configurable": {"thread_id": '1234'}}

initial_input = {
    "messages": [
        ("user", "Explain gradient descent in very simple terms.")
    ]
}

# Invoke the graph for the first time
result = app.invoke(initial_input, config=config)

In [15]:
print(result)

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='1b593266-a6c9-42b0-90c5-5dfc1485ec3e')], '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to answer a user question.', 'question': 'Explain gradient descent in very simple terms.', 'instruction': 'Approve this question? yes/no'}, id='863b1cd739b0388fc22108a78c65700d')]}


In [ ]:
"""
{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='1b593266-a6c9-42b0-90c5-5dfc1485ec3e')], '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to answer a user question.', 'question': 'Explain gradient descent in very simple terms.', 'instruction': 'Approve this question? yes/no'}, id='863b1cd739b0388fc22108a78c65700d')]}
"""

In [19]:
message = result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'Model is about to answer a user question.',
 'question': 'Explain gradient descent in very simple terms.',
 'instruction': 'Approve this question? yes/no'}

In [20]:
user_input = input(f"\nBackend message - {message} \n Approve this question? (y/n): ")

In [21]:
final_result = app.invoke(
    Command(resume={"approved": user_input}),
    config=config,
)

In [28]:
response = final_result['messages'][-1].content

In [29]:
print(response)

Imagine you are blindfolded and standing on a very bumpy, hilly landscape. Your goal is to get to the absolute lowest point in that valley.

**Gradient Descent is simply the method you use to find that lowest point.**

Here is a breakdown using simple terms:

---

### 1. The Setup (The Problem)

In the world of AI and Machine Learning, we are trying to teach a computer something by adjusting certain internal settings (we call these settings "parameters" or "weights").

*   **The Landscape:** The function you are trying to minimize is the "error" or the "cost." Think of this as the bumpy hill. Every point on the hill represents how wrong your current settings are.
*   **The Goal:** You want to find the very bottom of the valley—the lowest possible error score.

### 2. The Process (The Descent)

Since you are blindfolded, how do you know which way is down? You use the concept of the **Gradient**.

#### What is the Gradient?
The gradient is just a fancy word for the **slope** or the **ste